# NumPy Part 2 : advanced tools for AI
    1. einsum            ---> one notation for sums , products , transposes , batched matmul
    2. statistics        ---> percentiles , histogram , covariance , correlation , outliers
    3. linear algebra    ---> eigenvalues , SVD , rank , low-rank approximation , PCA
    4. windows & filters ---> sliding windows , convolution , padding , pooling
    5. grids             ---> meshgrid
    6. sets & repetition ---> isin , intersect1d , setdiff1d , repeat , tile
    7. frequencies       ---> FFT
    8. dates             ---> datetime64
    9. fitting           ---> polyfit , interp
   10. when you cannot vectorize ---> apply_along_axis , vectorize
   11. applications      ---> attention , PCA , recommender , forecasting , CNN layer , fault detection

In [1]:
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view
np.set_printoptions(precision=3, suppress=True)
print(np.__version__)

2.5.3


## 1. einsum : Einstein summation
`np.einsum("input_labels->output_labels", arrays...)`

    - every axis gets a LETTER
    - a letter that appears in the inputs but NOT in the output is SUMMED over
    - a letter in the output is KEPT , in the order you write it

In [2]:
a = np.array([1.0, 2.0, 3.0])
b = np.array([4.0, 5.0, 6.0])
M = np.arange(6.0).reshape(2, 3)

print("sum        ", np.einsum("i->", a), "=", a.sum())
print("dot        ", np.einsum("i,i->", a, b), "=", a @ b)
print("element    ", np.einsum("i,i->i", a, b), "=", a * b)
print("outer      \n", np.einsum("i,j->ij", a, b))
print("transpose  \n", np.einsum("ij->ji", M))
print("row sums   ", np.einsum("ij->i", M), "=", M.sum(axis=1))
print("col sums   ", np.einsum("ij->j", M), "=", M.sum(axis=0))

sum         6.0 = 6.0
dot         32.0 = 32.0
element     [ 4. 10. 18.] = [ 4. 10. 18.]
outer      
 [[ 4.  5.  6.]
 [ 8. 10. 12.]
 [12. 15. 18.]]
transpose  
 [[0. 3.]
 [1. 4.]
 [2. 5.]]
row sums    [ 3. 12.] = [ 3. 12.]
col sums    [3. 5. 7.] = [3. 5. 7.]


In [3]:
A = np.arange(6.0).reshape(2, 3)
B = np.arange(12.0).reshape(3, 4)
S = np.array([[1.0, 2.0], [3.0, 4.0]])
print("matmul ok :", np.allclose(np.einsum("ik,kj->ij", A, B), A @ B))
print("trace     :", np.einsum("ii->", S), "=", np.trace(S))
print("diagonal  :", np.einsum("ii->i", S))

# batched matrix multiplication : 10 independent (3x4)@(4x5) products at once
X = np.random.default_rng(0).normal(size=(10, 3, 4))
W = np.random.default_rng(1).normal(size=(10, 4, 5))
print("batched   :", np.einsum("bij,bjk->bik", X, W).shape, "| same as @ :", np.allclose(np.einsum("bij,bjk->bik", X, W), X @ W))

matmul ok : True
trace     : 5.0 = 5.0
diagonal  : [1. 4.]
batched   : (10, 3, 5) | same as @ : True


## 2. Statistics

In [4]:
rng = np.random.default_rng(42)
income = np.concatenate([rng.lognormal(mean=10.5, sigma=0.4, size=995), [900_000, 1_200_000, 2_000_000, 1_500_000, 950_000]])

print(f"mean   = {income.mean():>10,.0f}")
print(f"median = {np.median(income):>10,.0f}   <--- robust to the 5 very rich people")
print("quartiles (25,50,75):", np.percentile(income, [25, 50, 75]).round(0))
print("deciles :", np.quantile(income, np.linspace(0.1, 0.9, 9)).round(-3))

mean   =     45,136
median =     36,405   <--- robust to the 5 very rich people
quartiles (25,50,75): [27487. 36405. 46452.]
deciles : [22000. 25000. 30000. 33000. 36000. 40000. 43000. 50000. 61000.]


In [5]:
# histogram : counts per bin (the data behind a bar chart)
ages = rng.normal(35, 10, 1000).clip(18, 80)
counts, edges = np.histogram(ages, bins=[18, 25, 35, 45, 55, 65, 80])
for c, lo, hi in zip(counts, edges[:-1], edges[1:]):
    print(f"{lo:>3.0f}-{hi:<3.0f} {c:>4}  " + "#" * (c // 10))

 18-25   189  ##################
 25-35   329  ################################
 35-45   343  ##################################
 45-55   120  ############
 55-65    19  #
 65-80     0  


In [6]:
# outliers : z-score rule and IQR rule
z = (income - income.mean()) / income.std()
q1, q3 = np.percentile(income, [25, 75])
iqr = q3 - q1
iqr_mask = (income < q1 - 1.5 * iqr) | (income > q3 + 1.5 * iqr)
print("z-score > 3 :", (np.abs(z) > 3).sum(), "outliers")
print("IQR rule    :", iqr_mask.sum(), "outliers | upper fence =", round(q3 + 1.5 * iqr))
print("mean without IQR outliers:", round(income[~iqr_mask].mean()))

z-score > 3 : 5 outliers
IQR rule    : 36 outliers | upper fence = 74899
mean without IQR outliers: 37208


In [7]:
# covariance and correlation between features
n = 500
hours = rng.uniform(0, 10, n)                         # study hours
score = 50 + 4 * hours + rng.normal(0, 5, n)          # depends on hours
noise = rng.normal(0, 1, n)                           # unrelated
data = np.vstack([hours, score, noise])               # rows = variables

print("covariance matrix:\n", np.cov(data).round(2))
print("correlation matrix:\n", np.corrcoef(data).round(3))
print("weighted mean:", np.average([18, 15, 20], weights=[3, 2, 1]).round(3))

covariance matrix:
 [[  8.53  34.37   0.1 ]
 [ 34.37 163.99   0.52]
 [  0.1    0.52   0.96]]
correlation matrix:
 [[1.    0.919 0.035]
 [0.919 1.    0.041]
 [0.035 0.041 1.   ]]
weighted mean: 17.333


## 3. Eigenvalues , SVD , rank , low-rank approximation

In [8]:
C = np.array([[4.0, 2.0], [2.0, 3.0]])              # symmetric (like a covariance matrix)
vals, vecs = np.linalg.eigh(C)                       # eigh : for symmetric matrices (sorted , real)
print("eigenvalues :", vals)
print("eigenvectors (columns):\n", vecs)
v = vecs[:, 1]
print("C @ v == lambda * v :", np.allclose(C @ v, vals[1] * v))

eigenvalues : [1.438 5.562]
eigenvectors (columns):
 [[ 0.615 -0.788]
 [-0.788 -0.615]]
C @ v == lambda * v : True


In [9]:
# SVD : every matrix  A = U @ diag(S) @ Vt
A = np.array([[3.0, 1.0, 1.0], [-1.0, 3.0, 1.0]])
U, S, Vt = np.linalg.svd(A, full_matrices=False)
print("shapes:", U.shape, S.shape, Vt.shape)
print("singular values:", S)
print("reconstruction ok:", np.allclose(U @ np.diag(S) @ Vt, A))
print("rank:", np.linalg.matrix_rank(A), "| rank of [[1,2],[2,4]]:", np.linalg.matrix_rank(np.array([[1, 2], [2, 4]])))

shapes: (2, 2) (2,) (2, 3)
singular values: [3.464 3.162]
reconstruction ok: True
rank: 2 | rank of [[1,2],[2,4]]: 1


In [10]:
# low-rank approximation : keep only the k largest singular values
rng = np.random.default_rng(0)
true_low_rank = rng.normal(size=(100, 3)) @ rng.normal(size=(3, 80))     # rank 3
M = true_low_rank + 0.1 * rng.normal(size=(100, 80))                      # + noise
U, S, Vt = np.linalg.svd(M, full_matrices=False)
print("first singular values:", S[:6].round(1))
for k in (1, 2, 3, 10):
    Mk = U[:, :k] * S[:k] @ Vt[:k]                                        # (100,k)*(k,) @ (k,80)
    err = np.linalg.norm(M - Mk) / np.linalg.norm(M)
    params = k * (100 + 80 + 1)
    print(f"k={k:<3} relative error={err:.3f}  numbers stored={params:>5} vs {M.size}")

first singular values: [100.2  87.2  85.2   1.8   1.8   1.8]
k=1   relative error=0.773  numbers stored=  181 vs 8000
k=2   relative error=0.542  numbers stored=  362 vs 8000
k=3   relative error=0.055  numbers stored=  543 vs 8000
k=10  relative error=0.047  numbers stored= 1810 vs 8000


## 4. Sliding windows , convolution , padding , pooling

In [11]:
x = np.array([1, 2, 3, 4, 5, 6, 7])
print(sliding_window_view(x, 3))                      # every window of length 3 (a VIEW , no copy)
print("moving average:", sliding_window_view(x, 3).mean(axis=1))
print("np.convolve   :", np.convolve(x, np.ones(3) / 3, mode="valid"))

[[1 2 3]
 [2 3 4]
 [3 4 5]
 [4 5 6]
 [5 6 7]]
moving average: [2. 3. 4. 5. 6.]
np.convolve   : [2. 3. 4. 5. 6.]


In [12]:
# padding : add values around an array (keeps the output size after a convolution)
img = np.arange(1, 10).reshape(3, 3)
print(np.pad(img, 1))                                 # zeros around
print(np.pad(img, 1, mode="edge"))                    # repeat the border values

[[0 0 0 0 0]
 [0 1 2 3 0]
 [0 4 5 6 0]
 [0 7 8 9 0]
 [0 0 0 0 0]]
[[1 1 2 3 3]
 [1 1 2 3 3]
 [4 4 5 6 6]
 [7 7 8 9 9]
 [7 7 8 9 9]]


In [13]:
# 2D convolution with sliding_window_view + einsum  (what a CNN layer computes)
image = np.zeros((6, 6))
image[:, 3:] = 1.0                                    # left dark , right bright ---> a vertical edge
sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=float)

patches = sliding_window_view(np.pad(image, 1), (3, 3))     # (6, 6, 3, 3) : one 3x3 patch per pixel
response = np.einsum("ijkl,kl->ij", patches, sobel_x)       # multiply-and-sum every patch
print(image)
print(response)

[[0. 0. 0. 1. 1. 1.]
 [0. 0. 0. 1. 1. 1.]
 [0. 0. 0. 1. 1. 1.]
 [0. 0. 0. 1. 1. 1.]
 [0. 0. 0. 1. 1. 1.]
 [0. 0. 0. 1. 1. 1.]]
[[ 0.  0.  3.  3.  0. -3.]
 [ 0.  0.  4.  4.  0. -4.]
 [ 0.  0.  4.  4.  0. -4.]
 [ 0.  0.  4.  4.  0. -4.]
 [ 0.  0.  4.  4.  0. -4.]
 [ 0.  0.  3.  3.  0. -3.]]


In [14]:
# 2x2 max pooling with reshape : keep the strongest value in each 2x2 block
fm = np.arange(16).reshape(4, 4)
pooled = fm.reshape(2, 2, 2, 2).max(axis=(1, 3))
print(fm)
print(pooled)

[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]
 [12 13 14 15]]
[[ 5  7]
 [13 15]]


## 5. meshgrid : evaluate a function on a grid

In [15]:
xs = np.linspace(-3, 3, 7)
ys = np.linspace(-3, 3, 5)
XX, YY = np.meshgrid(xs, ys)                          # (5, 7) each
print(XX.shape, YY.shape)
print(XX)
print(YY)
Z = np.hypot(XX, YY) > 2                              # evaluate 'outside the circle' on every grid point
for row in Z[::-1]:
    print("".join("o" if v else "." for v in row))

(5, 7) (5, 7)
[[-3. -2. -1.  0.  1.  2.  3.]
 [-3. -2. -1.  0.  1.  2.  3.]
 [-3. -2. -1.  0.  1.  2.  3.]
 [-3. -2. -1.  0.  1.  2.  3.]
 [-3. -2. -1.  0.  1.  2.  3.]]
[[-3.  -3.  -3.  -3.  -3.  -3.  -3. ]
 [-1.5 -1.5 -1.5 -1.5 -1.5 -1.5 -1.5]
 [ 0.   0.   0.   0.   0.   0.   0. ]
 [ 1.5  1.5  1.5  1.5  1.5  1.5  1.5]
 [ 3.   3.   3.   3.   3.   3.   3. ]]
ooooooo
oo...oo
o.....o
oo...oo
ooooooo


## 6. Set operations , repeat , tile

In [16]:
train_ids = np.array([101, 102, 103, 104, 105])
test_ids = np.array([104, 106, 107, 101])
print("leak (intersect1d):", np.intersect1d(train_ids, test_ids))
print("only in test      :", np.setdiff1d(test_ids, train_ids))
print("all ids (union1d) :", np.union1d(train_ids, test_ids))
print("isin mask         :", np.isin(test_ids, train_ids))

cities = np.array(["Tehran", "Shiraz", "Paris", "Tabriz", "Berlin"])
print("Iranian cities    :", cities[np.isin(cities, ["Tehran", "Shiraz", "Tabriz", "Isfahan"])])

leak (intersect1d): [101 104]
only in test      : [106 107]
all ids (union1d) : [101 102 103 104 105 106 107]
isin mask         : [ True False False  True]
Iranian cities    : ['Tehran' 'Shiraz' 'Tabriz']


In [17]:
print(np.repeat([1, 2, 3], 2))                 # each element repeated
print(np.tile([1, 2, 3], 2))                   # the whole array repeated
print(np.tile(np.array([[0, 1]]), (3, 1)))     # 3 copies of a row

[1 1 2 2 3 3]
[1 2 3 1 2 3]
[[0 1]
 [0 1]
 [0 1]]


## 7. FFT : from time to frequency

In [18]:
fs = 1000                                              # samples per second
t = np.arange(0, 1, 1 / fs)                            # 1 second
signal = np.sin(2 * np.pi * 50 * t) + 0.5 * np.sin(2 * np.pi * 120 * t) + 0.3 * rng.normal(size=t.size)

spectrum = np.abs(np.fft.rfft(signal)) / (t.size / 2)  # amplitude of each frequency
freqs = np.fft.rfftfreq(t.size, d=1 / fs)
top = np.argsort(spectrum)[::-1][:3]
for i in sorted(top, key=lambda i: freqs[i]):
    print(f"{freqs[i]:6.1f} Hz  amplitude {spectrum[i]:.2f}")

  50.0 Hz  amplitude 1.01
 120.0 Hz  amplitude 0.50
 169.0 Hz  amplitude 0.05


## 8. Dates with datetime64

In [19]:
days = np.arange("2026-11-01", "2026-11-11", dtype="datetime64[D]")
print(days)
print("days until new year:", (np.datetime64("2027-01-01") - np.datetime64("2026-11-07")).astype(int))
weekday = (days.astype("datetime64[D]").view("int64") - 4) % 7      # 1970-01-01 was a Thursday ---> 0 = Monday
print("weekend mask (Sat/Sun):", np.isin(weekday, [5, 6]))
print(np.datetime64("2026-11-07T14:30") + np.timedelta64(90, "m"))

['2026-11-01' '2026-11-02' '2026-11-03' '2026-11-04' '2026-11-05'
 '2026-11-06' '2026-11-07' '2026-11-08' '2026-11-09' '2026-11-10']
days until new year: 55
weekend mask (Sat/Sun): [ True False False False False False  True  True False False]
2026-11-07T16:00


## 9. Polynomial fitting and interpolation

In [20]:
x = np.linspace(0, 4, 20)
y = 1 + 2 * x - 0.5 * x ** 2 + rng.normal(0, 0.1, x.size)
coef = np.polyfit(x, y, deg=2)                         # highest power first
print("fitted coefficients (x^2, x, 1):", coef.round(3))
print("prediction at x=5:", np.polyval(coef, 5).round(3))

# interpolation : estimate missing sensor values between two measurements
t_known = np.array([0, 10, 20, 30])
temp_known = np.array([20.0, 24.0, 23.0, 27.0])
print("interp at 5, 15, 25:", np.interp([5, 15, 25], t_known, temp_known))

fitted coefficients (x^2, x, 1): [-0.508  2.024  0.996]
prediction at x=5: -1.595
interp at 5, 15, 25: [22.  23.5 25. ]


## 10. When you cannot vectorize directly

In [21]:
import time
M = rng.integers(0, 5, size=(2000, 10))

def mode_of_row(row):
    return np.bincount(row).argmax()

t0 = time.perf_counter(); m1 = np.apply_along_axis(mode_of_row, 1, M); t1 = time.perf_counter() - t0
t0 = time.perf_counter()
counts = np.zeros((M.shape[0], 5), dtype=int)
np.add.at(counts, (np.repeat(np.arange(M.shape[0]), M.shape[1]), M.ravel()), 1)
m2 = counts.argmax(axis=1); t2 = time.perf_counter() - t0
print("same result:", np.array_equal(m1, m2), f"| apply_along_axis {t1 * 1000:.1f} ms | vectorized {t2 * 1000:.1f} ms")

# np.vectorize is only a convenience (a python loop inside) , NOT a speed-up
grade = np.vectorize(lambda s: "A" if s >= 18 else "B" if s >= 15 else "C")
print(grade(np.array([19.5, 16, 12])))
print(np.select([np.array([19.5, 16, 12]) >= 18, np.array([19.5, 16, 12]) >= 15], ["A", "B"], default="C"))

same result: True | apply_along_axis 5.9 ms | vectorized 1.1 ms
['A' 'B' 'C']
['A' 'B' 'C']


## 11. Applications in AI

In [22]:
# Application 1 : scaled dot-product attention (the heart of Transformers) with a causal mask
rng = np.random.default_rng(0)
tokens = ["I", "love", "deep", "learning"]
T, d_model, d_k = len(tokens), 8, 4
X = rng.normal(size=(T, d_model))                     # token embeddings
Wq, Wk, Wv = (rng.normal(size=(d_model, d_k)) for _ in range(3))
Q, K, V = X @ Wq, X @ Wk, X @ Wv                      # (T, d_k) each

scores = np.einsum("id,jd->ij", Q, K) / np.sqrt(d_k)  # (T, T) : how much token i looks at token j
causal = np.triu(np.ones((T, T), dtype=bool), k=1)    # True above the diagonal = the future
scores = np.where(causal, -np.inf, scores)            # a token may not look at future tokens

weights = np.exp(scores - scores.max(axis=1, keepdims=True))
weights /= weights.sum(axis=1, keepdims=True)         # softmax per row
out = np.einsum("ij,jd->id", weights, V)              # weighted average of the values

print("attention weights (rows sum to 1):")
print("          " + "  ".join(f"{t:>8}" for t in tokens))
for t, row in zip(tokens, weights):
    print(f"{t:>8}  " + "  ".join(f"{w:8.3f}" for w in row))
print("output shape:", out.shape)

attention weights (rows sum to 1):
                 I      love      deep  learning
       I     1.000     0.000     0.000     0.000
    love     0.970     0.030     0.000     0.000
    deep     0.915     0.000     0.085     0.000
learning     0.015     0.245     0.084     0.656
output shape: (4, 4)


In [23]:
# multi-head attention for a BATCH : one einsum handles batch (b) and heads (h)
b, h, T, d = 2, 3, 5, 4
Qb, Kb, Vb = (rng.normal(size=(b, h, T, d)) for _ in range(3))
S = np.einsum("bhid,bhjd->bhij", Qb, Kb) / np.sqrt(d)
P = np.exp(S - S.max(-1, keepdims=True)); P /= P.sum(-1, keepdims=True)
O = np.einsum("bhij,bhjd->bhid", P, Vb)
print("scores", S.shape, "| output", O.shape)

scores (2, 3, 5, 5) | output (2, 3, 5, 4)


In [24]:
# Application 2 : PCA from scratch with SVD
rng = np.random.default_rng(1)
n = 300
latent = rng.normal(size=(n, 2))                                  # 2 hidden factors
mixing = np.array([[2.0, 0.5, 1.0, 0.0, 1.5],
                   [0.0, 1.0, -1.0, 2.0, 0.5]])
X = latent @ mixing + 0.1 * rng.normal(size=(n, 5))               # 5 observed features

Xc = X - X.mean(axis=0)                                            # 1) center
U, S, Vt = np.linalg.svd(Xc, full_matrices=False)                  # 2) SVD
explained = S ** 2 / (S ** 2).sum()                                # 3) explained variance ratio
print("explained variance:", explained.round(4))
print("cumulative        :", np.cumsum(explained).round(4))

Z = Xc @ Vt[:2].T                                                  # 4) project to 2 dims
X_back = Z @ Vt[:2] + X.mean(axis=0)                               # 5) reconstruct
print("5 -> 2 dims , reconstruction error:", round(np.linalg.norm(X - X_back) / np.linalg.norm(X), 4))
print("covariance of components (diagonal = uncorrelated):\n", np.cov(Z.T).round(3))

explained variance: [0.549 0.448 0.001 0.001 0.001]
cumulative        : [0.549 0.997 0.998 0.999 1.   ]
5 -> 2 dims , reconstruction error: 0.0507
covariance of components (diagonal = uncorrelated):
 [[6.849 0.   ]
 [0.    5.583]]


In [25]:
# Application 3 : image compression with SVD
yy, xx = np.mgrid[0:64, 0:64]
img = (np.sin(xx / 6) + np.cos(yy / 9) + ((xx - 32) ** 2 + (yy - 32) ** 2 < 150)) * 80 + 40
img = img.clip(0, 255)
U, S, Vt = np.linalg.svd(img, full_matrices=False)
for k in (1, 3, 5, 10):
    approx = U[:, :k] * S[:k] @ Vt[:k]
    err = np.abs(img - approx).mean()
    ratio = img.size / (k * (64 + 64 + 1))
    print(f"rank {k:>2}: mean abs error {err:6.2f} | {ratio:5.1f}x smaller")

rank  1: mean abs error  11.13 |  31.8x smaller
rank  3: mean abs error   4.14 |  10.6x smaller
rank  5: mean abs error   2.75 |   6.4x smaller
rank 10: mean abs error   1.00 |   3.2x smaller


In [26]:
# Application 4 : a tiny recommender with low-rank matrix completion (the Netflix-prize idea)
R = np.array([[5, 4, 0, 1, 0],
              [4, 0, 4, 1, 1],
              [0, 5, 5, 0, 1],
              [1, 1, 0, 5, 4],
              [0, 1, 1, 4, 5],
              [1, 0, 1, 5, 0]], dtype=float)          # 0 = not rated
users = ["Ali", "Sara", "Reza", "Mina", "Omid", "Neda"]
movies = ["Action1", "Action2", "Action3", "Drama1", "Drama2"]
known = R > 0

filled = np.where(known, R, R.sum(0) / known.sum(0))       # start : missing ---> movie mean
for _ in range(50):
    U, S, Vt = np.linalg.svd(filled, full_matrices=False)
    low = U[:, :2] * S[:2] @ Vt[:2]                         # rank-2 : 'action taste' and 'drama taste'
    filled = np.where(known, R, low)                        # keep the real ratings , update the missing ones

pred = low.clip(1, 5)
for u, name in enumerate(users):
    missing = np.flatnonzero(~known[u])
    recs = ", ".join(f"{movies[j]}={pred[u, j]:.1f}" for j in missing)
    print(f"{name:<5} predicted: {recs}")

Ali   predicted: Action3=4.4, Drama2=1.0
Sara  predicted: Action2=3.6
Reza  predicted: Action1=5.0, Drama1=1.0
Mina  predicted: Action3=1.0
Omid  predicted: Action1=1.0
Neda  predicted: Action2=1.0, Drama2=5.0


In [27]:
# Application 5 : time-series forecasting with sliding windows + least squares (an autoregressive model)
rng = np.random.default_rng(2)
t = np.arange(240)
series = 10 + 0.05 * t + 3 * np.sin(2 * np.pi * t / 24) + rng.normal(0, 0.5, t.size)   # trend + daily cycle

w = 24
windows = sliding_window_view(series, w + 1)            # (216, 25) : 24 inputs + 1 target
X, y = windows[:, :-1], windows[:, -1]
split = 180
Xb = np.hstack([X, np.ones((len(X), 1))])
coef = np.linalg.lstsq(Xb[:split], y[:split], rcond=None)[0]

pred = Xb[split:] @ coef
naive = X[split:, -1]                                    # baseline : 'tomorrow = today'
rmse = lambda a, b: np.sqrt(np.mean((a - b) ** 2))
print(f"test RMSE  AR model : {rmse(pred, y[split:]):.3f}")
print(f"test RMSE  naive    : {rmse(naive, y[split:]):.3f}")

test RMSE  AR model : 0.685
test RMSE  naive    : 0.982


In [28]:
# Application 6 : a convolution layer + ReLU + max pooling (forward pass of a tiny CNN)
rng = np.random.default_rng(3)
batch = rng.random((4, 8, 8))                           # 4 grayscale images 8x8
kernels = rng.normal(size=(6, 3, 3))                    # 6 filters 3x3

patches = sliding_window_view(batch, (3, 3), axis=(1, 2))           # (4, 6, 6, 3, 3)
feature_maps = np.einsum("bijkl,fkl->bfij", patches, kernels)       # (4, 6 filters, 6, 6)
activated = np.maximum(feature_maps, 0)                             # ReLU
pooled = activated.reshape(4, 6, 3, 2, 3, 2).max(axis=(3, 5))       # 2x2 max pooling
flat = pooled.reshape(4, -1)                                         # ready for a dense layer
print("patches", patches.shape, "-> conv", feature_maps.shape, "-> pool", pooled.shape, "-> flat", flat.shape)

patches (4, 6, 6, 3, 3) -> conv (4, 6, 6, 6) -> pool (4, 6, 3, 3) -> flat (4, 54)


In [29]:
# Application 7 : vibration fault detection with FFT features + an expert rule
fs, n = 2000, 2000
t = np.arange(n) / fs
rng = np.random.default_rng(4)

def machine_signal(faulty):
    s = np.sin(2 * np.pi * 30 * t) + 0.2 * rng.normal(size=n)      # 30 Hz : shaft rotation
    if faulty:
        s += 0.4 * np.sin(2 * np.pi * 157 * t)                        # 157 Hz : bearing defect frequency
    return s

signals = np.stack([machine_signal(faulty=i % 3 == 0) for i in range(9)])    # (9 machines , 2000 samples)
amp = np.abs(np.fft.rfft(signals, axis=1)) / (n / 2)
freqs = np.fft.rfftfreq(n, 1 / fs)
band = (freqs > 150) & (freqs < 165)
band_energy = amp[:, band].max(axis=1)

alarm = band_energy > 0.2
for i, (e, a) in enumerate(zip(band_energy, alarm)):
    print(f"machine {i}: 150-165 Hz amplitude {e:.3f}  {'<--- BEARING ALARM' if a else ''}")

machine 0: 150-165 Hz amplitude 0.399  <--- BEARING ALARM
machine 1: 150-165 Hz amplitude 0.021  
machine 2: 150-165 Hz amplitude 0.017  
machine 3: 150-165 Hz amplitude 0.400  <--- BEARING ALARM
machine 4: 150-165 Hz amplitude 0.015  
machine 5: 150-165 Hz amplitude 0.015  
machine 6: 150-165 Hz amplitude 0.401  <--- BEARING ALARM
machine 7: 150-165 Hz amplitude 0.018  
machine 8: 150-165 Hz amplitude 0.013  


## Exercises
1. with `einsum` write : the sum of all elements of a matrix , the row-wise dot product of two (n,d) matrices , and a batched outer product.
2. remove the causal mask in Application 1 : how do the attention weights of the FIRST token change ? why ?
3. find the income outliers with a z-score threshold of 2 instead of 3 : how many ? why is the IQR rule more robust here ?
4. in Application 2 , how many components do you need to keep 99 % of the variance ?
5. use `sliding_window_view` to compute a 7-day moving maximum of a random daily series.
6. change the kernel in the convolution cell to detect HORIZONTAL edges.
7. in Application 5 , try window sizes 6 , 12 and 48 : which one is best and why ?
8. in Application 7 , make the defect weaker (0.15) : does the alarm still work ? how would you choose the threshold ?